# `ptof_obs_backtest_thresholds`

## Purpose (Iteration 1 of the backtest plan)
For each of the 16 detectors, answer: **"if this threshold had been live for the full retained
bronze history, how often would it have fired, and how close were the near-misses?"** This is
a distribution question over history, not a lifecycle replay -- it does not simulate job
cadence, dedup, notify cooldown, or digest routing (see `ptof_obs_backtest_replay.ipynb` for
that). Confirms/refutes claims already recorded in `threshold_basis`/handoff docs (e.g.
`etl_run_slow`'s "5 real events," `capability_silence`'s "0 empirical false positives") with a
full-history sweep instead of a one-off snapshot.

## What this notebook does NOT do
- Does not touch `obs_incidents` (real or backtest) -- no incident lifecycle here at all.
- Does not write to any production detector table (`capability_silence`,
  `etl_run_slow`, etc.) -- every detector's `CREATE OR REPLACE TABLE ... AS` builder is
  stripped down to its bare `SELECT` before being run, so the sweep is read-only against the
  bronze views and never overwrites a live findings table.
- Skips `unacknowledged_critical` / `long_running_incident` -- both operate on `obs_incidents`
  itself, not raw bronze, so there is no bronze-only historical signal to sweep here. They are
  covered by `ptof_obs_backtest_replay.ipynb` instead, against the shadow incidents table.

## How detector SQL is sourced
Every detector notebook (`ptof_obs_liveness_detection.ipynb`, `ptof_obs_mal_output.ipynb`,
`ptof_obs_behavioral_correlation.ipynb`) was refactored (2026-09-22, backtest support) so each
detector's condition lives in a `def <detector>_sql(as_of="current_timestamp()"): ...` builder,
called with the default from the notebook's own cell so production behavior is unchanged. This
notebook `%run`s those three notebooks to import the builder functions directly -- the single
source of truth for each detector's logic, so this sweep can never drift out of sync with what
actually runs in production.

**One documented side effect of the `%run`s below:** each of those three notebooks ends its own
cells by calling its builder(s) with the default `as_of` and persisting the result to its real
production table (e.g. `capability_silence`, `blank_output_findings`) -- exactly what the
scheduled `obs_fresh_scan` job already does on its normal cadence. Running this notebook by
hand therefore triggers one extra, harmless, idempotent recompute of those tables using
*current* data (not historical) -- it does not touch `obs_incidents`, does not notify Teams,
and does not use stale/backtest data for any production table. `ptof_obs_alert.ipynb` is
deliberately **not** `%run` here for the same reason in reverse: its cells write to the real
`obs_incidents` table and POST to the live Teams webhook, which this notebook must never
trigger. The 3 in-scope scalar checks defined in its cell 4 (`pipeline_heartbeat`,
`etl_pipeline_staleness`, `nightly_baseline_staleness` -- all bronze/baseline-table reads, no
`obs_incidents` dependency) are duplicated locally below instead, with a comment pointing back
at that cell as the source of truth to keep in sync by hand.

## Mechanics
- **Row-grain detectors** (most of the 16): evaluated once per day, stepped across the full
  min/max timestamp range actually present in the relevant bronze view (queried directly, not
  assumed) via a Python loop over calendar days calling each detector's `_sql(as_of=...)`
  builder (stripped to its bare `SELECT`) at each step.
- **`pipeline_heartbeat`** (25-minute window): stepped hourly instead of daily, since a daily
  step would almost never land inside its own window.
- **`etl_run_slow`** (MAD/percentile-based): stepped weekly, and instead of joining today's
  single `etl_duration_baseline` snapshot, this notebook recomputes a **rolling** 30-day
  baseline ending at each historical step (reusing the exact SQL from
  `ptof_obs_nightly_baseline.ipynb` cell 2, `{as_of}`-parameterized the same way) -- otherwise
  the backtest would be unfairly informed by future data the real detector would not have had
  at that point in history.
- Output: one row per `(detector, as_of)` with fire/no-fire, margin (where the detector has a
  natural numeric threshold), and detail -- persisted to
  `mq_gmdf_dev.oil_obs.backtest_threshold_sweep`.


In [ ]:
CAT = "mq_gmdf_dev.oil_obs"
import json
from datetime import datetime, timedelta, timezone
import re

print(f"backtest threshold sweep -- target catalog: {CAT}")

In [ ]:
# Import the real detector builder functions -- the single source of truth for each
# detector's condition. See markdown cell above for why ptof_obs_alert.ipynb is deliberately
# NOT %run here (it writes to real obs_incidents and posts to the live Teams webhook).
%run "./ptof_obs_liveness_detection"

In [ ]:
%run "./ptof_obs_mal_output"

In [ ]:
%run "./ptof_obs_behavioral_correlation"

In [ ]:
# Local copies of the 3 in-scope scalar checks from ptof_obs_alert.ipynb cell 4
# (pipeline_heartbeat, etl_pipeline_staleness, nightly_baseline_staleness) -- duplicated rather
# than %run because that notebook's other cells write to the real obs_incidents table and POST
# to the live Teams webhook, which this backtest must never trigger. unacknowledged_critical
# and long_running_incident are intentionally NOT duplicated here -- they read obs_incidents
# itself, not raw bronze, so there is no bronze-only historical signal for Iteration 1 to sweep
# (see ptof_obs_backtest_replay.ipynb for those two against the shadow incidents table).
#
# Keep these in sync BY HAND with ptof_obs_alert.ipynb cell 4 if that cell's SQL ever changes --
# this is the one deliberate exception to the "single source of truth" design used everywhere
# else in this notebook.

def pipeline_heartbeat_sql(as_of="current_timestamp()"):
    return f"""
    SELECT CASE WHEN count(*) = 0 THEN 1 ELSE 0 END AS n, count(*) AS rows_last_25m
    FROM {CAT}.v_llm_bronze
    WHERE called_at >= {as_of} - INTERVAL 25 MINUTES
    """

def etl_pipeline_staleness_sql(as_of="current_timestamp()"):
    return f"""SELECT CASE WHEN max(run_timestamp) < {as_of} - INTERVAL 30 MINUTES
                    THEN 1 ELSE 0 END AS n,
           max(run_timestamp) AS latest_run
        FROM {CAT}.v_etl_bronze"""

def nightly_baseline_staleness_sql(as_of="current_timestamp()"):
    return f"""SELECT count(*) AS n, max(computed_at) AS last_computed
        FROM {CAT}.response_field_baseline
        WHERE computed_at < {as_of} - INTERVAL 36 HOURS"""

print("Local scalar-check builders defined (pipeline_heartbeat, etl_pipeline_staleness, "
      "nightly_baseline_staleness).")

In [ ]:
# --- Sweep infrastructure -----------------------------------------------------------------

_CREATE_TABLE_RE = re.compile(r"(?is)^\s*CREATE OR REPLACE TABLE\s+\S+\s+AS\s*")


def select_body(sql_text):
    """Strip a `_sql()` builder's `CREATE OR REPLACE TABLE ... AS` preamble, leaving a bare
    SELECT that can be run read-only without ever touching the real production table. Every
    builder in this pipeline follows this exact convention; scalar checks (which are already
    bare SELECTs) pass through unchanged."""
    return _CREATE_TABLE_RE.sub("", sql_text).strip()


def as_of_literal(dt):
    """Python datetime -> a Spark SQL TIMESTAMP literal safe to substitute for {as_of}."""
    return "TIMESTAMP'{}'".format(dt.strftime("%Y-%m-%d %H:%M:%S"))


def bronze_bounds(table, ts_col):
    row = spark.sql(f"SELECT min({ts_col}) AS lo, max({ts_col}) AS hi FROM {CAT}.{table}").first()
    return row.lo, row.hi


def daily_steps(lo, hi):
    if lo is None or hi is None:
        return []
    cur = lo.replace(hour=0, minute=0, second=0, microsecond=0)
    end = hi.replace(hour=0, minute=0, second=0, microsecond=0)
    steps = []
    while cur <= end:
        steps.append(cur)
        cur += timedelta(days=1)
    return steps


def hourly_steps(lo, hi):
    if lo is None or hi is None:
        return []
    cur = lo.replace(minute=0, second=0, microsecond=0)
    end = hi.replace(minute=0, second=0, microsecond=0)
    steps = []
    while cur <= end:
        steps.append(cur)
        cur += timedelta(hours=1)
    return steps


def weekly_steps(lo, hi):
    if lo is None or hi is None:
        return []
    cur = lo.replace(hour=0, minute=0, second=0, microsecond=0)
    end = hi.replace(hour=0, minute=0, second=0, microsecond=0)
    steps = []
    while cur <= end:
        steps.append(cur)
        cur += timedelta(days=7)
    return steps


_sweep_rows = []  # accumulates (detector, as_of, fired, margin, detail)


def record_step(detector, as_of_dt, sql_text, margin_sql_text=None):
    """Run a detector's stripped SELECT at one as_of step; append one sweep row.

    fired = the detector's own condition returned >=1 row at this as_of (exactly the same
    fire/no-fire semantics the production CREATE OR REPLACE TABLE would have produced).
    margin: when margin_sql_text is given, it must return a single row with a `margin` column
    (positive = past threshold, negative = headroom) computed across ALL rows regardless of
    whether the HAVING/WHERE clause would have fired -- i.e. "how close was the tightest
    row/capability/table to firing", not just whether the detector as a whole fired. Left NULL
    for detectors with no natural continuous margin (config/binary checks).
    """
    df = spark.sql(select_body(sql_text))
    rows = df.limit(5).collect()
    fired = len(rows) > 0
    detail = json.dumps([r.asDict() for r in rows], default=str)[:2000]

    margin = None
    if margin_sql_text is not None:
        mrow = spark.sql(margin_sql_text).first()
        if mrow is not None:
            margin = mrow["margin"]

    _sweep_rows.append((detector, as_of_dt, fired, margin, detail))


print("Sweep infrastructure ready.")

In [ ]:
# --- Row-grain detectors, daily cadence ------------------------------------------------------

lo, hi = bronze_bounds("v_llm_bronze", "called_at")
llm_days = daily_steps(lo, hi)
print(f"v_llm_bronze range: {lo} .. {hi} -> {len(llm_days)} daily steps")

for as_of_dt in llm_days:
    as_of = as_of_literal(as_of_dt)

    # capability_silence: margin = worst (hours_since_last_call - silence_grace_hours) across
    # covered capabilities -- positive means some capability is past its own grace window.
    record_step(
        "capability_silence", as_of_dt, capability_silence_sql(as_of),
        margin_sql_text=f"""
            SELECT max(
                round((unix_timestamp({as_of}) - unix_timestamp(max(b.called_at)))/3600.0, 1)
                - r.silence_grace_hours
            ) AS margin
            FROM {CAT}.capability_registry r
            LEFT JOIN {CAT}.v_llm_bronze b
                   ON b.capability = r.capability AND b.called_at >= {as_of} - INTERVAL 7 DAYS
            WHERE r.active = true AND r.silence_grace_hours IS NOT NULL
            GROUP BY r.capability, r.silence_grace_hours
        """,
    )

    # capability_silence_ceiling: same shape, ceiling instead of grace.
    record_step(
        "capability_silence_ceiling", as_of_dt, capability_silence_ceiling_sql(as_of),
        margin_sql_text=f"""
            SELECT max(
                round((unix_timestamp({as_of}) - unix_timestamp(max(b.called_at)))/3600.0, 1)
                - r.silence_ceiling_hours
            ) AS margin
            FROM {CAT}.capability_registry r
            LEFT JOIN {CAT}.v_llm_bronze b ON b.capability = r.capability
            WHERE r.active = true AND r.silence_grace_hours IS NULL
              AND r.silence_ceiling_hours IS NOT NULL
            GROUP BY r.capability, r.silence_ceiling_hours
        """,
    )

    # capability_liveness_unconfigured: pure config check, no numeric margin.
    record_step("capability_liveness_unconfigured", as_of_dt,
                capability_liveness_unconfigured_sql(as_of))

    # shift_context_missing: count-based, no natural continuous margin.
    record_step("shift_context_missing", as_of_dt, shift_context_missing_sql(as_of))

    # blank_output_findings: margin = worst blank_count_window - 1 (its HAVING floor).
    record_step(
        "blank_output", as_of_dt, blank_output_findings_sql(as_of),
        margin_sql_text=f"""
            SELECT max(blank_count_window) - 1 AS margin
            FROM (
                {select_body(blank_output_findings_sql(as_of)).replace(
                    "HAVING sum(blank_output_count) >= 1", "HAVING true")}
            )
        """,
    )

    # response_schema_drift (schema_field_missing): binary presence check, no continuous margin.
    record_step("schema_field_missing", as_of_dt, response_schema_drift_sql(as_of))

print(f"llm-bronze daily sweep complete: {len(_sweep_rows)} rows so far.")

In [ ]:
# --- ETL row-grain detectors, daily cadence --------------------------------------------------

lo, hi = bronze_bounds("v_etl_bronze", "run_timestamp")
etl_days = daily_steps(lo, hi)
print(f"v_etl_bronze range: {lo} .. {hi} -> {len(etl_days)} daily steps")

for as_of_dt in etl_days:
    as_of = as_of_literal(as_of_dt)

    record_step("etl_pipeline_health", as_of_dt, etl_pipeline_health_sql(as_of))

    # etl_table_staleness: margin = worst minutes_since_last_run - 60 across all tables.
    record_step(
        "etl_table_staleness", as_of_dt, etl_table_staleness_sql(as_of),
        margin_sql_text=f"""
            SELECT max(round((unix_timestamp({as_of}) - unix_timestamp(max(run_timestamp)))/60.0, 1) - 60) AS margin
            FROM {CAT}.v_etl_bronze
            GROUP BY table_or_view
        """,
    )

    # etl_row_count_anomaly: regime crossing, binary, no continuous margin.
    record_step("etl_row_count_anomaly", as_of_dt, etl_row_count_anomaly_sql(as_of))

    # etl_pipeline_staleness (scalar, from ptof_obs_alert.ipynb cell 4): margin = minutes since
    # last fleet-wide run minus the 30-minute threshold.
    record_step(
        "etl_pipeline_staleness", as_of_dt, etl_pipeline_staleness_sql(as_of),
        margin_sql_text=f"""
            SELECT round((unix_timestamp({as_of}) - unix_timestamp(max(run_timestamp)))/60.0, 1) - 30 AS margin
            FROM {CAT}.v_etl_bronze
        """,
    )

print(f"etl-bronze daily sweep complete: {len(_sweep_rows)} rows so far.")

In [ ]:
# --- Handover detectors (v_ish_bronze), daily cadence ----------------------------------------

lo, hi = bronze_bounds("v_ish_bronze", "ts")
ish_days = daily_steps(lo, hi)
print(f"v_ish_bronze range: {lo} .. {hi} -> {len(ish_days)} daily steps")

for as_of_dt in ish_days:
    as_of = as_of_literal(as_of_dt)

    record_step("handover_delivery", as_of_dt, handover_delivery_failures_sql(as_of))

    # handover_delivery_rate: margin = max(failure_pct_7d - 20, (failed - 3) as a pct-equivalent
    # is not comparable, so report the rate-path margin; the absolute-count OR-trigger is
    # already reflected in `fired` via the builder's own WHERE clause).
    record_step(
        "handover_delivery_rate", as_of_dt, handover_delivery_rate_sql(as_of),
        margin_sql_text=f"""
            SELECT round(count_if(after_v:sent::boolean = false AND is_email_disabled_gate = false) * 100.0
                  / nullif(count_if(after_v:sent::boolean = true)
                           + count_if(after_v:sent::boolean = false AND is_email_disabled_gate = false), 0), 1)
                  - 20 AS margin
            FROM {CAT}.v_ish_bronze
            WHERE entity_type = 'HandoverEmail' AND ts >= {as_of} - INTERVAL 7 DAYS
        """,
    )

print(f"ish-bronze daily sweep complete: {len(_sweep_rows)} rows so far.")

In [ ]:
# --- pipeline_heartbeat, hourly cadence (its own window is 25 minutes -- a daily step would
# almost never land inside it) ----------------------------------------------------------------

lo, hi = bronze_bounds("v_llm_bronze", "called_at")
llm_hours = hourly_steps(lo, hi)
print(f"pipeline_heartbeat hourly sweep: {len(llm_hours)} steps over v_llm_bronze range "
      f"{lo} .. {hi}")

for as_of_dt in llm_hours:
    as_of = as_of_literal(as_of_dt)
    record_step(
        "pipeline_heartbeat", as_of_dt, pipeline_heartbeat_sql(as_of),
        margin_sql_text=f"""
            SELECT count(*) AS margin
            FROM {CAT}.v_llm_bronze
            WHERE called_at >= {as_of} - INTERVAL 25 MINUTES
        """,
    )

# nightly_baseline_staleness: its own window (36h) is coarser than daily -- hourly stepping adds
# no information a daily step would not already capture, so it stays out of this cell. Swept
# daily below instead, over response_field_baseline's own computed_at range.
lo_b, hi_b = bronze_bounds("response_field_baseline", "computed_at")
baseline_days = daily_steps(lo_b, hi_b)
print(f"nightly_baseline_staleness daily sweep: {len(baseline_days)} steps over "
      f"response_field_baseline range {lo_b} .. {hi_b}")

for as_of_dt in baseline_days:
    as_of = as_of_literal(as_of_dt)
    record_step(
        "nightly_baseline_staleness", as_of_dt, nightly_baseline_staleness_sql(as_of),
        margin_sql_text=f"""
            SELECT round((unix_timestamp({as_of}) - unix_timestamp(max(computed_at)))/3600.0, 1) - 36 AS margin
            FROM {CAT}.response_field_baseline
        """,
    )

print(f"pipeline_heartbeat/nightly_baseline_staleness sweep complete: {len(_sweep_rows)} rows so far.")

In [ ]:
# --- etl_run_slow, weekly cadence, ROLLING 30-day baseline per step -------------------------
#
# etl_run_slow depends on etl_duration_baseline, which is a single snapshot recomputed nightly
# from a trailing 30-day window ending at "now". Reusing that live table directly in a
# historical sweep would let the detector see anomaly bounds computed from data AFTER the
# as_of step being tested -- lookahead bias. Instead, at each weekly step, recompute the exact
# same median/MAD logic from ptof_obs_nightly_baseline.ipynb cell 2 with {as_of} substituted for
# current_timestamp() (30-day window ending at that step), materialize it into a throwaway temp
# view, and join etl_run_slow's own condition against THAT instead of the live table.

def rolling_etl_duration_baseline_sql(as_of):
    """Adapted from ptof_obs_nightly_baseline.ipynb cell 2 -- same median/MAD/upper_bound_s
    logic, {as_of}-parameterized (2 substitutions: the 30-day WHERE filter and computed_at)
    instead of current_timestamp(), so the baseline only reflects data available as of this
    historical step."""
    return f"""
    WITH eligible AS (
        SELECT table_or_view, task_name, duration_seconds
        FROM {CAT}.v_etl_bronze
        WHERE run_timestamp >= {as_of} - INTERVAL 30 DAYS
          AND run_timestamp < {as_of}
          AND status = 'success'
    ),
    med AS (
        SELECT table_or_view, task_name,
               count(*)                                     AS n,
               percentile_approx(duration_seconds, 0.5)      AS med_s
        FROM eligible
        GROUP BY table_or_view, task_name
    )
    SELECT
        e.table_or_view,
        e.task_name,
        m.n,
        m.med_s                                                        AS median_duration_s,
        percentile_approx(abs(e.duration_seconds - m.med_s), 0.5)      AS mad_duration_s,
        greatest(
            m.med_s + 5 * 1.4826 * percentile_approx(abs(e.duration_seconds - m.med_s), 0.5),
            m.med_s * 1.5
        )                                                               AS upper_bound_s,
        {as_of}                                                        AS computed_at
    FROM eligible e
    JOIN med m USING (table_or_view, task_name)
    GROUP BY e.table_or_view, e.task_name, m.n, m.med_s
    HAVING m.n >= 50
    """


def etl_run_slow_sql_rolling(as_of):
    """Same flagging logic as etl_run_slow_sql() (ptof_obs_liveness_detection.ipynb cell 9), but
    joined against the freshly-recomputed rolling baseline view instead of the live
    etl_duration_baseline table."""
    return f"""
    WITH flagged AS (
        SELECT
            e.table_or_view, e.task_name, e.duration_seconds, e.run_timestamp,
            window(e.run_timestamp, '60 minutes').start AS window_start,
            base.upper_bound_s
        FROM {CAT}.v_etl_bronze e
        JOIN _rolling_etl_duration_baseline base
          ON base.table_or_view = e.table_or_view AND base.task_name = e.task_name
        WHERE e.run_timestamp >= {as_of} - INTERVAL 24 HOURS
          AND e.run_timestamp < {as_of}
          AND e.status = 'success'
          AND e.duration_seconds > base.upper_bound_s
    )
    SELECT
        task_name, window_start,
        count(*)                                       AS anomalous_count,
        count(distinct table_or_view)                  AS affected_table_count,
        concat_ws(', ', collect_set(table_or_view))     AS affected_tables,
        max(duration_seconds)                          AS max_duration_s,
        max(upper_bound_s)                             AS upper_bound_s,
        {as_of}                                        AS detected_at
    FROM flagged
    GROUP BY task_name, window_start
    HAVING count(*) >= 3
    """


lo, hi = bronze_bounds("v_etl_bronze", "run_timestamp")
etl_weeks = weekly_steps(lo, hi)
print(f"etl_run_slow weekly rolling-baseline sweep: {len(etl_weeks)} steps over "
      f"v_etl_bronze range {lo} .. {hi}")

for as_of_dt in etl_weeks:
    as_of = as_of_literal(as_of_dt)
    spark.sql(rolling_etl_duration_baseline_sql(as_of)).createOrReplaceTempView(
        "_rolling_etl_duration_baseline")
    record_step(
        "etl_run_slow", as_of_dt, etl_run_slow_sql_rolling(as_of),
        margin_sql_text=f"""
            SELECT max(e.duration_seconds - base.upper_bound_s) AS margin
            FROM {CAT}.v_etl_bronze e
            JOIN _rolling_etl_duration_baseline base
              ON base.table_or_view = e.table_or_view AND base.task_name = e.task_name
            WHERE e.run_timestamp >= {as_of} - INTERVAL 24 HOURS AND e.run_timestamp < {as_of}
              AND e.status = 'success'
        """,
    )

print(f"etl_run_slow sweep complete: {len(_sweep_rows)} total rows accumulated.")

In [ ]:
# --- Persist the sweep -----------------------------------------------------------------------

sweep_schema = "detector STRING, as_of TIMESTAMP, fired BOOLEAN, margin DOUBLE, detail STRING"
sweep_df = spark.createDataFrame(
    [(d, ts, bool(f), (float(m) if m is not None else None), det)
     for d, ts, f, m, det in _sweep_rows],
    schema=sweep_schema,
)
sweep_df.write.mode("overwrite").saveAsTable(f"{CAT}.backtest_threshold_sweep")
print(f"Wrote {sweep_df.count()} rows to {CAT}.backtest_threshold_sweep")

In [ ]:
# --- Per-detector summary ----------------------------------------------------------------

spark.sql(f"""
    SELECT
        detector,
        count(*)                                   AS steps_evaluated,
        sum(int(fired))                             AS steps_fired,
        round(sum(int(fired)) * 100.0 / count(*), 2) AS fire_rate_pct,
        round(min(margin), 2)                       AS min_margin,
        round(max(margin), 2)                       AS max_margin,
        round(avg(margin), 2)                       AS avg_margin
    FROM {CAT}.backtest_threshold_sweep
    GROUP BY detector
    ORDER BY fire_rate_pct DESC
""").show(20, truncate=False)

## Verdict (fill in after a live run)

This notebook has never been executed against live Databricks/Spark in this session -- there is
no live connection available in the environment that authored it. Every query above has been
written to be syntactically and semantically correct against the schemas documented in
`TECHNICAL_REFERENCE.md` and the source detector notebooks, and the `{as_of}` substitutions were
verified textually against each detector's pre-refactor literal query (see the refactor commit),
but **none of it has actually run**, so no fire-rate numbers exist yet to spot-check against
`threshold_basis`/handoff docs.

**Spot-checks to run once this executes for real** (per the plan's Verification section):
- `capability_silence` -- expect ~0% fire rate / large negative margins, matching the "0
  empirical false positives" already recorded.
- `etl_run_slow` -- expect roughly 5 fired weeks (or fewer, since rolling-baseline recomputation
  is stricter than the live single-snapshot baseline) matching the "5 real events in a week"
  note, not "5 events total over the whole sweep."
- `capability_silence_ceiling` -- expect 0 fires (has never fired on a real event as of
  2026-09-22, per `ptof_obs_liveness_detection.ipynb`).

**Known design deviations from a literal reading of the plan, both documented above and worth
restating here:**
- The plan suggested implementing the daily/hourly sweep as "one SQL query with a generated
  calendar table" rather than a Python loop calling each `_sql()` builder per step. This
  notebook uses the Python loop instead, so it can call the exact same builder functions the
  production detectors use (zero drift risk) rather than re-deriving each condition in a
  different, calendar-joined SQL shape. This is more expensive per detector but was judged a
  better match for the plan's own stated design decision ("extract each detector's condition
  ... callable from both the real detector notebook and the backtest notebook").
- Several detectors (`capability_liveness_unconfigured`, `shift_context_missing`,
  `etl_pipeline_health`, `etl_row_count_anomaly`, `schema_field_missing`,
  `handover_delivery`) have no natural continuous margin -- they are configuration or binary
  presence/absence checks, not threshold-vs-actual comparisons -- so `margin` is left NULL for
  those rows. `fired`/`detail` are still populated for all 16 in-scope-minus-2 detectors.
- `etl_pipeline_staleness` and `nightly_baseline_staleness` (scalar checks whose builder
  functions live in `ptof_obs_alert.ipynb` cell 4) are duplicated locally in this notebook
  rather than imported via `%run`, since `%run`-ing that notebook wholesale would execute its
  Teams-posting and real-`obs_incidents`-writing cells -- a side effect this backtest must never
  trigger. These local copies must be kept in sync by hand if that cell's SQL ever changes.
